In [1]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass, asdict
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans

# Week 12 additions (PCA-guided structure)
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

# =========================================================
# WEEK 12 — FUNCTION 6 (PCA-GUIDED + CLUSTER-AWARE BO)
# Goal: propose x_next in [0,1]^5 (<= 6 decimals) targeting local maxima.
#
# Week 12 upgrades (PCA lens):
#  1) PCA on standardized X to identify variance-dominant directions (PC subspace)
#  2) Candidate generation shifted to:
#       - PCA-Sobol (global but in PC subspace)
#       - PCA trust region around best observed
#       - PCA trust region around best cluster centroid
#       - Directed "PC ascent" small-step grid (structured exploitation)
#       - Far random reduced (redundancy removal)
#  3) Adaptive min_dist based on dataset density (pairwise p10)
#  4) Keep Week 11: robust qEI + diversity penalty + soft cluster bonus
# =========================================================

# -----------------------------
# 1) Data (as provided)
# -----------------------------
X_raw = np.array([
 [0.7281861, 0.15469257, 0.73255167, 0.69399651, 0.05640131],
 [0.24238435, 0.84409997, 0.5778091, 0.67902128, 0.50195289],
 [0.72952261, 0.7481062, 0.67977464, 0.35655228, 0.67105368],
 [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
 [0.6188123, 0.33180214, 0.18728787, 0.75623847, 0.3288348],
 [0.78495809, 0.91068235, 0.7081201, 0.95922543, 0.0049115],
 [0.14511079, 0.8966846, 0.89632223, 0.72627154, 0.23627199],
 [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
 [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
 [0.75759436, 0.35583141, 0.0165229, 0.4342072, 0.11243304],
 [0.5367969, 0.30878091, 0.41187929, 0.38822518, 0.5225283],
 [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
 [0.6293079, 0.80348368, 0.81140844, 0.04561319, 0.11062446],
 [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
 [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
 [0.25890557, 0.79367771, 0.6421139, 0.19667346, 0.59310318],
 [0.43216593, 0.71561781, 0.3418191, 0.70499988, 0.61496184],
 [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
 [0.9217762, 0.93187122, 0.41487637, 0.59505727, 0.73562569],
 [0.12667892, 0.2914703, 0.06452848, 0.6805146, 0.89281919],
 [1.057739, 1.031871, 1.078805, 1.061655, 0.992819],
 [0.183405, 0.304243, 0.524756, 0.431945, 0.29123 ],
 [0.268807, 0.268756, 0.495982, 0.986904, 0.010463],
 [0.071886, 0.119564, 0.11427 , 0.97486 , 0.062381],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062],
 [0.289326, 0.014308, 0.819185, 0.769045, 0.018673],
 [0.437318, 0.251829, 0.542272, 0.788906, 0.000000],
 [0.395320, 0.238886, 0.702458, 0.805605, 0.000000],
 [0.019634, 0.099529, 0.422099, 0.998212, 0.080632],
 [0.414000, 0.246000, 0.780000, 0.832000, 0.000000]
], dtype=float)

y_raw = np.array([
 -0.71426495, -1.20995524, -1.67219994, -1.53605771, -0.82923655,
 -1.24704893, -1.23378638, -1.69434344, -2.57116963, -1.30911635,
 -1.14478485, -1.91267714, -1.62283895, -1.35668211, -2.0184254,
 -1.70255784, -1.29424696, -0.93575656, -2.15576776, -1.74688209,
 -2.868905011263093, -0.9489046340640067, -0.6674108573004914,
 -1.3769650251311083, -2.5104529076756172, -2.5498833751068073,
 -0.7529123509459484, -0.36323892495106164, -0.29318472382773614,
 -1.1654038633763473, -0.36523877995026865
], dtype=float)

assert X_raw.shape[0] == y_raw.shape[0], "X and y must have the same number of rows."
assert X_raw.shape[1] == 5, "Function 6 expected dimension=5."

# -----------------------------
# 2) Device & Seeds (reproducibility)
# -----------------------------
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 3) Data hygiene (clamp + dedup)
# -----------------------------
def clamp01(X):
    return np.clip(X, 0.0, 1.0)

def dedup_average(X, y, tol=0.0):
    """
    If duplicate X rows exist, average their y values.
    tol=0.0 => exact duplicates only (byte-wise).
    """
    if tol > 0:
        X_key = np.round(X / tol) * tol
    else:
        X_key = X.copy()

    keys = [row.tobytes() for row in X_key]
    buckets = {}
    for i, k in enumerate(keys):
        buckets.setdefault(k, []).append(i)

    X_new, y_new = [], []
    for _, idxs in buckets.items():
        X_new.append(X[idxs[0]])
        y_new.append(float(np.mean(y[idxs])))
    return np.array(X_new, dtype=float), np.array(y_new, dtype=float), buckets

X_raw = clamp01(X_raw)
X_raw, y_raw, dedup_buckets = dedup_average(X_raw, y_raw, tol=0.0)

# -----------------------------
# 4) Scaling
# -----------------------------
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X_raw)
y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32, device=device)

# -----------------------------
# 4b) PCA view (Week 12)
# -----------------------------
def fit_pca_view(X_obs, y_obs, var_threshold=0.92, seed=123):
    """
    Fit PCA on standardized X. Keep k PCs reaching cumulative variance threshold.
    Fit Ridge on PC scores (in original y space) to estimate an ascent direction in PC space.
    """
    Xs = x_scaler.transform(X_obs)
    pca = PCA(random_state=seed).fit(Xs)
    cum = np.cumsum(pca.explained_variance_ratio_)
    k = int(np.searchsorted(cum, var_threshold) + 1)
    k = int(np.clip(k, 2, X_obs.shape[1]))

    Z = pca.transform(Xs)[:, :k]
    ridge = Ridge(alpha=1.0, random_state=seed).fit(Z, y_obs)

    d = ridge.coef_.astype(float)
    if np.linalg.norm(d) < 1e-12:
        d = np.ones_like(d)
    d = d / (np.linalg.norm(d) + 1e-12)

    return pca, k, d, pca.explained_variance_ratio_, cum

# -----------------------------
# 5) Surrogate Model
# -----------------------------
class SurrogateNN(nn.Module):
    def __init__(self, input_dim=5, hidden_layers=(128, 64), dropout_p=0.05, use_layernorm=True):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            if use_layernorm:
                layers.append(nn.LayerNorm(h))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout_p))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# -----------------------------
# 6) Training helper
# -----------------------------
def train_with_early_stopping(model, optimizer, criterion,
                              X_train, y_train, X_val, y_val,
                              max_epochs=1200, patience=180, min_epochs=120):
    best_state = None
    best_val = float("inf")
    no_improve = 0

    for ep in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            if ep >= min_epochs:
                no_improve += 1

        if ep >= min_epochs and no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return best_val

# -----------------------------
# 7) Hyperparameter tuning (Successive Halving)
# -----------------------------
def sample_log_uniform(rng, lo, hi):
    return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

@dataclass(frozen=True)
class BOConfig:
    hidden_layers: tuple
    dropout: float
    lr: float
    weight_decay: float
    use_layernorm: bool
    n_ensemble: int
    xi_base: float

def sample_config(rng) -> BOConfig:
    hidden_choices = [(64, 64), (128, 64), (128, 128), (256, 128)]
    dropout_choices = [0.0, 0.03, 0.06, 0.10, 0.16]
    ln_choices = [True, False]
    return BOConfig(
        hidden_layers=hidden_choices[rng.integers(0, len(hidden_choices))],
        dropout=float(dropout_choices[rng.integers(0, len(dropout_choices))]),
        lr=sample_log_uniform(rng, 4e-4, 4e-3),
        weight_decay=sample_log_uniform(rng, 1e-7, 3e-4),
        use_layernorm=bool(rng.choice(ln_choices)),
        n_ensemble=int(rng.choice([5, 7])),
        xi_base=float(rng.choice([0.0, 0.005, 0.01, 0.02])),
    )

def cv_score_config(cfg: BOConfig, X_all, y_all, seed=123, n_splits=4,
                    max_epochs=600, patience=120, min_epochs=80):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    criterion = nn.MSELoss()
    vals = []
    for tr_idx, va_idx in kf.split(X_all):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        model = SurrogateNN(
            input_dim=5,
            hidden_layers=cfg.hidden_layers,
            dropout_p=cfg.dropout,
            use_layernorm=cfg.use_layernorm
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        val = train_with_early_stopping(
            model, optimizer, criterion,
            X_tr, y_tr, X_va, y_va,
            max_epochs=max_epochs, patience=patience, min_epochs=min_epochs
        )
        vals.append(val)
    return float(np.mean(vals))

def tune_hyperparameters_successive_halving(X_all, y_all, seed=123):
    rng = np.random.default_rng(seed)

    stages = [
        {"n_cfg": 18, "keep": 6, "epochs": 350, "splits": 3},
        {"n_cfg": 6,  "keep": 2, "epochs": 650, "splits": 4},
        {"n_cfg": 2,  "keep": 1, "epochs": 1100, "splits": 5},
    ]

    cfgs = [sample_config(rng) for _ in range(stages[0]["n_cfg"])]

    stage_logs = []
    for si, st in enumerate(stages, 1):
        scores = []
        for i, cfg in enumerate(cfgs, 1):
            sc = cv_score_config(
                cfg, X_all, y_all, seed=seed,
                n_splits=st["splits"],
                max_epochs=st["epochs"],
                patience=max(90, int(0.20 * st["epochs"])),
                min_epochs=max(60, int(0.12 * st["epochs"]))
            )
            scores.append(sc)
            print(f"[SH stage {si}] cfg {i:02d}/{len(cfgs)} | CV-MSE={sc:.6f} | {asdict(cfg)}")

        order = np.argsort(scores)
        kept = [cfgs[j] for j in order[:st["keep"]]]
        best_sc = float(scores[order[0]])
        best_cfg = kept[0]
        stage_logs.append({
            "stage": si,
            "epochs": st["epochs"],
            "splits": st["splits"],
            "kept": st["keep"],
            "best_cv_mse": best_sc,
            "best_cfg": asdict(best_cfg),
        })

        print(f"\n[SH stage {si}] best so far: {best_sc:.6f} | {asdict(best_cfg)}\n")
        cfgs = kept

    return best_cfg, best_sc, stage_logs

# -----------------------------
# 8) Deep Ensemble fit + predict
# -----------------------------
def fit_ensemble(cfg: BOConfig, X_all, y_all, base_seed=123):
    models = []
    criterion = nn.MSELoss()
    n = X_all.shape[0]

    for m in range(cfg.n_ensemble):
        rng = np.random.default_rng(base_seed + 10_000 + m)
        boot = rng.integers(0, n, size=n)
        X_b = X_all[boot]
        y_b = y_all[boot]

        idx = torch.randperm(n, device=device)
        n_train = max(int(0.85 * n), 1)
        tr_idx, va_idx = idx[:n_train], idx[n_train:]

        model = SurrogateNN(
            input_dim=5,
            hidden_layers=cfg.hidden_layers,
            dropout_p=cfg.dropout,
            use_layernorm=cfg.use_layernorm
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        train_with_early_stopping(
            model, optimizer, criterion,
            X_b[tr_idx], y_b[tr_idx],
            X_b[va_idx], y_b[va_idx],
            max_epochs=1200, patience=180, min_epochs=120
        )

        models.append(model)

    return models

@torch.no_grad()
def predict_ensemble(models, X_candidates):
    X_scaled_cand = x_scaler.transform(X_candidates)
    X_t = torch.tensor(X_scaled_cand, dtype=torch.float32, device=device)

    preds_scaled = []
    for model in models:
        model.eval()
        preds_scaled.append(model(X_t).detach().cpu().numpy())  # [N,1]

    preds_scaled = np.stack(preds_scaled, axis=0).squeeze(-1)  # [E, N]
    flat = preds_scaled.reshape(-1, 1)
    orig = y_scaler.inverse_transform(flat).reshape(preds_scaled.shape)

    mu = orig.mean(axis=0)
    sigma = orig.std(axis=0)
    return mu, sigma, orig

# -----------------------------
# 9) Acquisition (qEI) + helpers
# -----------------------------
def qei_mc(preds_members, y_best, xi=0.0, n_mc=96, seed=123):
    rng = np.random.default_rng(seed)
    E, N = preds_members.shape
    idx = rng.integers(0, E, size=(n_mc, N))
    samples = preds_members[idx, np.arange(N)[None, :]]  # [n_mc, N]
    improv = np.maximum(samples - (y_best + xi), 0.0)
    return improv.mean(axis=0)

def pairwise_min_dist(X_cand, X_obs):
    diff = X_cand[:, None, :] - X_obs[None, :, :]
    d2 = np.sum(diff * diff, axis=2)
    dmin = np.sqrt(np.min(d2, axis=1) + 1e-12)
    return dmin

def diversity_penalty_from_dmin(dmin, scale=0.02):
    return np.exp(-dmin / max(scale, 1e-6))

# -----------------------------
# 10) Candidate generation helpers
# -----------------------------
def sobol_candidates(n, dim, seed=123):
    engine = torch.quasirandom.SobolEngine(dimension=dim, scramble=True, seed=seed)
    return engine.draw(n).cpu().numpy()

def trust_region_candidates(x_center, n, sigma=0.08, seed=123):
    rng = np.random.default_rng(seed)
    X = x_center.reshape(1, -1) + rng.normal(0.0, sigma, size=(n, x_center.size))
    return np.clip(X, 0.0, 1.0)

def cluster_region_candidates(x_center, n, sigma_vec, seed=123):
    rng = np.random.default_rng(seed)
    X = x_center.reshape(1, -1) + rng.normal(0.0, 1.0, size=(n, x_center.size)) * sigma_vec.reshape(1, -1)
    return np.clip(X, 0.0, 1.0)

# Week 12: PCA-space candidates
def pca_sobol_candidates(pca, k, n, seed=123, z_clip=2.25):
    engine = torch.quasirandom.SobolEngine(dimension=k, scramble=True, seed=seed)
    U = engine.draw(n).cpu().numpy()          # [0,1]
    Z = (U * 2.0 - 1.0) * z_clip              # [-z_clip, +z_clip]

    Xs = (Z @ pca.components_[:k, :]) + pca.mean_
    X = x_scaler.inverse_transform(Xs)
    return np.clip(X, 0.0, 1.0)

def pca_trust_candidates(pca, k, z_center, n, sigma=0.30, seed=123):
    rng = np.random.default_rng(seed)
    Z = z_center.reshape(1, -1) + rng.normal(0.0, sigma, size=(n, k))
    Xs = (Z @ pca.components_[:k, :]) + pca.mean_
    X = x_scaler.inverse_transform(Xs)
    return np.clip(X, 0.0, 1.0)

# -----------------------------
# 11) Clustering diagnostics (Week 11, kept)
# -----------------------------
def cluster_diagnostics_kmeans(X_obs, y_obs, k=4, seed=123):
    Xs = x_scaler.transform(X_obs)
    km = KMeans(n_clusters=k, random_state=seed, n_init=20)
    labels = km.fit_predict(Xs)

    centroids_scaled = km.cluster_centers_
    centroids_orig = x_scaler.inverse_transform(centroids_scaled)

    summary = []
    for c in range(k):
        idx = np.where(labels == c)[0]
        if len(idx) == 0:
            continue
        summary.append({
            "cluster": int(c),
            "n": int(len(idx)),
            "y_mean": float(np.mean(y_obs[idx])),
            "y_max": float(np.max(y_obs[idx])),
            "best_x": X_obs[idx[np.argmax(y_obs[idx])]].astype(float).tolist(),
        })

    best_cluster = max(summary, key=lambda d: (d["y_max"], d["y_mean"]))["cluster"]
    return labels, centroids_orig, summary, int(best_cluster)

# -----------------------------
# 14) Data gap diagnostics (bias / coverage)
# -----------------------------
def dataset_diagnostics(X, y):
    diag = {}
    diag["n_points"] = int(X.shape[0])
    diag["y_best"] = float(np.max(y))
    diag["y_worst"] = float(np.min(y))
    diag["y_mean"] = float(np.mean(y))
    diag["y_std"] = float(np.std(y))

    per_dim = []
    for j in range(X.shape[1]):
        xj = X[:, j]
        per_dim.append({
            "dim": j + 1,
            "min": float(np.min(xj)),
            "max": float(np.max(xj)),
            "mean": float(np.mean(xj)),
            "std": float(np.std(xj)),
            "near_0": int(np.sum(xj <= 0.05)),
            "near_1": int(np.sum(xj >= 0.95)),
        })
    diag["per_dim"] = per_dim

    if X.shape[0] >= 2:
        diff = X[:, None, :] - X[None, :, :]
        d = np.sqrt(np.sum(diff * diff, axis=2) + 1e-12)
        d = d[~np.eye(X.shape[0], dtype=bool)]
        diag["pairwise_dist_mean"] = float(np.mean(d))
        diag["pairwise_dist_p10"] = float(np.quantile(d, 0.10))
        diag["pairwise_dist_p50"] = float(np.quantile(d, 0.50))
        diag["pairwise_dist_p90"] = float(np.quantile(d, 0.90))
    else:
        diag["pairwise_dist_mean"] = None

    return diag

# -----------------------------
# 12) Propose next point (Week 12: PCA-guided + cluster-aware)
# -----------------------------
def propose_next_point(models, X_obs, y_obs,
                       xi_base=0.01,
                       n_sobol=22000, n_local=9000,
                       random_seed=123,
                       min_dist=1e-3,             # used as fallback; adaptive min_dist overrides
                       local_sigma=0.085,         # kept for compatibility (not primary in Week 12)
                       top_k=10,
                       hedge_frac=0.05,
                       hedge_mode="ucb",
                       n_ref=12, alpha=0.7,
                       k_clusters=4,
                       cluster_radius=0.15,
                       pca_var_threshold=0.92
                       ):
    n = X_obs.shape[0]
    y_best = float(np.max(y_obs))
    y_min = float(np.min(y_obs))
    y_range = max(y_best - y_min, 1e-6)
    x_best_obs = X_obs[int(np.argmax(y_obs))]

    scale_n = float((n_ref / max(n, 1.0)) ** alpha)
    scale_n = float(np.clip(scale_n, 0.35, 1.25))

    # ---- Clustering view (kept)
    labels, centroids_orig, cl_summary, best_cluster = cluster_diagnostics_kmeans(
        X_obs, y_obs, k=k_clusters, seed=random_seed
    )
    x_cluster_center = np.clip(centroids_orig[best_cluster], 0.0, 1.0)

    # ---- Week 12: PCA view + ascent direction
    pca, k_pcs, pc_ascent_dir, pca_var, pca_cum = fit_pca_view(
        X_obs, y_obs, var_threshold=pca_var_threshold, seed=random_seed
    )

    # PC coords of best and best cluster centroid
    x_best_scaled = x_scaler.transform(x_best_obs.reshape(1, -1))
    z_best = pca.transform(x_best_scaled)[:, :k_pcs].ravel()

    x_cluster_scaled = x_scaler.transform(x_cluster_center.reshape(1, -1))
    z_cluster = pca.transform(x_cluster_scaled)[:, :k_pcs].ravel()

    # ---- Candidate mix (PCA-guided; redundancy reduced)
    X_pca_global = pca_sobol_candidates(pca, k_pcs, int(0.60 * n_sobol), seed=random_seed, z_clip=2.25)
    X_pca_best = pca_trust_candidates(pca, k_pcs, z_best, int(0.70 * n_local), sigma=0.28, seed=random_seed + 11)
    X_pca_cluster = pca_trust_candidates(pca, k_pcs, z_cluster, int(0.45 * n_local), sigma=0.33, seed=random_seed + 19)

    # Directed PC steps (structured exploitation)
    step_grid = np.linspace(0.05, 0.45, 9)
    Z_steps = np.array([z_best + t * pc_ascent_dir for t in step_grid], dtype=float)
    Xs_steps = (Z_steps @ pca.components_[:k_pcs, :]) + pca.mean_
    X_steps = np.clip(x_scaler.inverse_transform(Xs_steps), 0.0, 1.0)

    # Far random reduced
    rng = np.random.default_rng(random_seed + 999)
    X_far = rng.random(size=(250, 5))

    X_cand = np.vstack([X_pca_global, X_pca_best, X_pca_cluster, X_steps, X_far])
    source_tags = (
        ["pca_sobol"] * len(X_pca_global) +
        ["pca_best"] * len(X_pca_best) +
        ["pca_cluster"] * len(X_pca_cluster) +
        ["pc_step"] * len(X_steps) +
        ["far_random"] * len(X_far)
    )
    source_tags = np.array(source_tags)

    # ---- Adaptive min_dist (PCA redundancy removal idea)
    diag_tmp = dataset_diagnostics(X_obs, y_obs)
    p10 = diag_tmp.get("pairwise_dist_p10", 0.03)
    min_dist_adapt = float(np.clip(0.35 * p10, 8e-4, 4e-3))

    # Filter near-duplicates
    dmin_all = pairwise_min_dist(X_cand, X_obs)
    keep = dmin_all >= min_dist_adapt
    X_cand = X_cand[keep]
    source_tags = source_tags[keep]
    dmin = dmin_all[keep]

    # Predict
    mu, sigma, preds_members = predict_ensemble(models, X_cand)

    # uncertainty-aware xi
    sigma_med = float(np.median(sigma))
    conf_scale = 1.0 / (1.0 + sigma_med)
    effective_xi = float(xi_base * y_range * conf_scale * scale_n)

    # MAIN: robust qEI + diversity penalty
    qei = qei_mc(preds_members, y_best=y_best, xi=effective_xi, n_mc=96, seed=random_seed + 33)
    pen = diversity_penalty_from_dmin(dmin, scale=0.02)

    # Cluster proximity bonus (soft bias toward best cluster)
    d_to_cluster = np.linalg.norm(X_cand - x_cluster_center.reshape(1, -1), axis=1)
    cluster_bonus = np.exp(-d_to_cluster / max(cluster_radius, 1e-6))

    score_main = qei * (1.0 - 0.35 * pen) * (0.85 + 0.15 * cluster_bonus)

    # HEDGE
    if hedge_mode == "ucb":
        beta = 2.5 + 2.0 * scale_n
        score_hedge = mu + beta * sigma
    else:
        score_hedge = sigma

    rng2 = np.random.default_rng(random_seed + 202)
    do_hedge = (rng2.random() < hedge_frac)

    if do_hedge:
        best_idx = int(np.argmax(score_hedge))
        chosen_mode = "HEDGE"
        chosen_score = float(score_hedge[best_idx])
    else:
        best_idx = int(np.argmax(score_main))
        chosen_mode = "MAIN(qEI+cluster+PCA)"
        chosen_score = float(score_main[best_idx])

    top_idx = np.argsort(-score_main)[:top_k]
    uniq = np.unique(source_tags)
    src_counts = {k: int(np.sum(source_tags == k)) for k in uniq}

    return {
        "x_next": X_cand[best_idx],
        "mu_next": float(mu[best_idx]),
        "sigma_next": float(sigma[best_idx]),
        "qei_next": float(qei[best_idx]),
        "dmin_next": float(dmin[best_idx]),
        "source_next": str(source_tags[best_idx]),
        "chosen_mode": chosen_mode,
        "chosen_score": chosen_score,
        "x_best_obs": x_best_obs,
        "y_best_obs": y_best,
        "effective_xi": float(effective_xi),
        "min_dist_adapt": float(min_dist_adapt),
        "n_candidates": int(X_cand.shape[0]),
        "source_counts": src_counts,
        "pca": {
            "k": int(k_pcs),
            "explained_var_ratio": np.round(pca_var.astype(float), 6).tolist(),
            "cum_explained": np.round(pca_cum.astype(float), 6).tolist(),
            "var_threshold": float(pca_var_threshold),
            "pc_ascent_dir": np.round(pc_ascent_dir.astype(float), 6).tolist(),
        },
        "cluster": {
            "k": int(k_clusters),
            "best_cluster_id": int(best_cluster),
            "best_cluster_centroid": np.round(x_cluster_center.astype(float), 6).tolist(),
            "cluster_radius": float(cluster_radius),
            "summary": cl_summary,
        },
        "topk": [
            {
                "rank": int(r + 1),
                "x": X_cand[i],
                "mu": float(mu[i]),
                "sigma": float(sigma[i]),
                "qei": float(qei[i]),
                "score": float(score_main[i]),
                "dmin": float(dmin[i]),
                "cluster_bonus": float(cluster_bonus[i]),
                "source": str(source_tags[i]),
            }
            for r, i in enumerate(top_idx)
        ],
        "assumptions": [
            "Locality/smoothness: best region likely near current best -> exploit via PCA trust region.",
            "High-variance directions (PCs) contain most useful movement -> global exploration in PC subspace reduces redundancy.",
            "Ridge on PC scores gives a weak ascent cue -> small structured steps help final-stage exploitation.",
            "Observed clusters indicate repeatable regions -> centroid-biased bonus remains useful.",
            "Objective uses bounded inputs; clamping to [0,1] is valid.",
        ]
    }

# -----------------------------
# 13) Interpretability utilities
# -----------------------------
def local_sensitivity_single_model(model, x_point):
    model.eval()
    x_scaled = x_scaler.transform(x_point.reshape(1, -1))
    x_t = torch.tensor(x_scaled, dtype=torch.float32, device=device, requires_grad=True)

    y_pred = model(x_t)
    y_pred.backward()

    grads = x_t.grad.detach().cpu().numpy().flatten()
    g = np.abs(grads)
    if g.sum() == 0:
        return np.ones_like(g) / len(g)
    return g / g.sum()

def local_sensitivity_ensemble(models, x_point):
    sens = np.stack([local_sensitivity_single_model(m, x_point) for m in models], axis=0)
    return sens.mean(axis=0), sens.std(axis=0)

def one_at_a_time_probe(models, x_point, delta=0.02):
    x0 = x_point.copy()
    out = []
    for d in range(x0.size):
        xp = x0.copy(); xp[d] = np.clip(xp[d] + delta, 0.0, 1.0)
        xm = x0.copy(); xm[d] = np.clip(xm[d] - delta, 0.0, 1.0)
        mu_p, _, _ = predict_ensemble(models, xp.reshape(1, -1))
        mu_m, _, _ = predict_ensemble(models, xm.reshape(1, -1))
        out.append({
            "dim": d + 1,
            "x_minus": float(xm[d]),
            "x_plus": float(xp[d]),
            "mu_minus": float(mu_m[0]),
            "mu_plus": float(mu_p[0]),
            "delta_mu": float(mu_p[0] - mu_m[0]),
        })
    return out

# -----------------------------
# 15) Main
# -----------------------------
def main():
    print("\n================ WEEK 12 FUNCTION 6 — PCA-GUIDED NEXT DATA POINT (<= 6 DECIMALS) ================\n")

    audit = {
        "seed": RANDOM_SEED,
        "device": str(device),
        "n_after_clamp_dedup": int(X_raw.shape[0]),
        "dedup_groups": int(len(dedup_buckets)),
        "notes": [
            "Objective: maximise y. Clamp to [0,1] and average exact duplicates.",
            "Week 12: PCA-guided candidate generation in variance-dominant PC subspace + directed PC ascent steps.",
            "Keep Week 11: KMeans cluster bonus + robust qEI + diversity penalty.",
            "Redundancy reduction: far-random shrunk; adaptive min_dist based on pairwise p10."
        ]
    }

    diag = dataset_diagnostics(X_raw, y_raw)
    audit["dataset_diagnostics"] = diag

    print("Data after clamp+dedup:", f"n={X_raw.shape[0]} points")
    print("Best observed y:", round(diag["y_best"], 6))

    print("\n================ HYPERPARAMETER TUNING (SUCCESSIVE HALVING) ================\n")
    best_cfg, best_cv, sh_logs = tune_hyperparameters_successive_halving(
        X_tensor_all, y_tensor_all, seed=RANDOM_SEED
    )
    audit["tuning"] = {"best_cv_mse_scaled_y": float(best_cv), "best_cfg": asdict(best_cfg), "sh_logs": sh_logs}

    print("\n================ FITTING DEEP ENSEMBLE ================\n")
    ensemble_models = fit_ensemble(best_cfg, X_tensor_all, y_tensor_all, base_seed=RANDOM_SEED)

    # Candidate budgets (keep transparent)
    n = X_raw.shape[0]
    n_sobol = int(np.clip(26000 - 500 * (n - 12), 16000, 26000))
    n_local = int(np.clip(9000 + 300 * (n - 12), 7000, 12000))
    audit["candidate_budget"] = {"n_sobol": n_sobol, "n_local": n_local, "n_far": 250}

    details = propose_next_point(
        ensemble_models,
        X_raw,
        y_raw,
        xi_base=best_cfg.xi_base,
        n_sobol=n_sobol,
        n_local=n_local,
        random_seed=RANDOM_SEED,
        min_dist=1e-3,
        local_sigma=0.085,
        top_k=10,
        hedge_frac=0.05,
        hedge_mode="ucb",
        n_ref=12,
        alpha=0.7,
        k_clusters=4,
        cluster_radius=0.15,
        pca_var_threshold=0.92
    )

    x_next = np.clip(details["x_next"].astype(float), 0.0, 1.0)
    x_next_6 = np.round(x_next, 6)

    # interpretability
    sens_mu, sens_sd = local_sensitivity_ensemble(ensemble_models, x_next)
    probe = one_at_a_time_probe(ensemble_models, x_next, delta=0.02)

    audit["proposal"] = {
        "x_best_obs": np.round(details["x_best_obs"].astype(float), 6).tolist(),
        "y_best_obs": float(details["y_best_obs"]),
        "x_next": x_next_6.tolist(),
        "mu_next": float(details["mu_next"]),
        "sigma_next": float(details["sigma_next"]),
        "qei_next": float(details["qei_next"]),
        "effective_xi": float(details["effective_xi"]),
        "chosen_mode": details["chosen_mode"],
        "chosen_score": float(details["chosen_score"]),
        "nearest_dist_to_data": float(details["dmin_next"]),
        "adaptive_min_dist": float(details["min_dist_adapt"]),
        "candidate_source": details["source_next"],
        "pca": details["pca"],
        "cluster": details["cluster"],
        "assumptions": details["assumptions"],
    }
    audit["interpretability"] = {
        "local_sensitivity_mean": np.round(sens_mu, 4).tolist(),
        "local_sensitivity_std": np.round(sens_sd, 4).tolist(),
        "oaat_probe_delta": 0.02,
        "oaat_probe": probe,
    }

    # -----------------------------
    # PRINT RESULTS (what you submit)
    # -----------------------------
    print("\n================ NEXT DATA POINT (SUBMIT THIS) — <= 6 DECIMALS ================\n")
    print("x_next =", x_next_6)

    print("\n================ MODEL PREDICTION (SURROGATE) ================\n")
    print("mu(x_next)    =", round(details["mu_next"], 6))
    print("sigma(x_next) =", round(details["sigma_next"], 6))
    print("qEI_MC        =", round(details["qei_next"], 6))

    print("\n================ WHY THIS POINT (PCA + CLUSTER) ================\n")
    print(f"chosen_mode   = {details['chosen_mode']}")
    print(f"source        = {details['source_next']}")
    print(f"effective_xi  = {details['effective_xi']:.6f}")
    print(f"adaptive_min_dist = {details['min_dist_adapt']:.6f}")
    print(f"nearest_dist  = {details['dmin_next']:.6f}")
    print("PCA: k=", details["pca"]["k"], "var_threshold=", details["pca"]["var_threshold"])
    print("PCA explained var ratio:", details["pca"]["explained_var_ratio"])
    print("cluster(best): id=", details["cluster"]["best_cluster_id"], "centroid=", details["cluster"]["best_cluster_centroid"])
    print("cluster summary:")
    for row in details["cluster"]["summary"]:
        print("  -", row)

    print("\n================ TOP-10 SHORTLIST (by MAIN score) — 6 DECIMALS ================\n")
    for row in details["topk"]:
        print(
            f"{row['rank']:02d}) x={np.round(row['x'].astype(float), 6)} | "
            f"mu={row['mu']:.6f} | sigma={row['sigma']:.6f} | qEI={row['qei']:.6f} | "
            f"score={row['score']:.6f} | bonus={row['cluster_bonus']:.3f} | "
            f"dmin={row['dmin']:.6f} | src={row['source']}"
        )

    print("\n================ LOCAL SENSITIVITY (ENSEMBLE GRADIENTS) ================\n")
    for i, (m, s) in enumerate(zip(sens_mu, sens_sd), 1):
        print(f"Dim {i}: mean={m:.3f} | std={s:.3f}")

    print("\n================ ONE-AT-A-TIME PROBE (delta=0.02) ================\n")
    for row in probe:
        print(f"Dim {row['dim']}: mu(+)= {row['mu_plus']:.6f} | mu(-)= {row['mu_minus']:.6f} | delta_mu= {row['delta_mu']:.6f}")

    print("\n================ DATASET GAPS / BIASES (DIAGNOSTICS) ================\n")
    print(f"n_points={diag['n_points']} | y_best={diag['y_best']:.6f} | y_mean={diag['y_mean']:.6f} | y_std={diag['y_std']:.6f}")
    print("Pairwise distances:",
          f"mean={diag.get('pairwise_dist_mean', None)}",
          f"p10={diag.get('pairwise_dist_p10', None)}",
          f"p50={diag.get('pairwise_dist_p50', None)}",
          f"p90={diag.get('pairwise_dist_p90', None)}")
    for d in diag["per_dim"]:
        print(f"Dim {d['dim']}: min={d['min']:.3f} max={d['max']:.3f} mean={d['mean']:.3f} std={d['std']:.3f} near0={d['near_0']} near1={d['near_1']}")

    print("\n================ KEY ASSUMPTIONS (EXPLICIT) ================\n")
    for a in details["assumptions"]:
        print("-", a)

    # Optional: machine-readable record
    # import json
    # print("\nAUDIT_JSON=\n", json.dumps(audit, indent=2))

if __name__ == "__main__":
    main()


================ WEEK 12 FUNCTION 6 — PCA-GUIDED NEXT DATA POINT (<= 6 DECIMALS) ================

Data after clamp+dedup: n=30 points
Best observed y: -0.293185

================ HYPERPARAMETER TUNING (SUCCESSIVE HALVING) ================

[SH stage 1] cfg 01/18 | CV-MSE=0.465374 | {'hidden_layers': (64, 64), 'dropout': 0.1, 'lr': 0.00045277351007438286, 'weight_decay': 5.837380423441397e-07, 'use_layernorm': True, 'n_ensemble': 5, 'xi_base': 0.005}
[SH stage 1] cfg 02/18 | CV-MSE=0.139432 | {'hidden_layers': (64, 64), 'dropout': 0.03, 'lr': 0.003352779476383228, 'weight_decay': 9.15551370547805e-07, 'use_layernorm': False, 'n_ensemble': 7, 'xi_base': 0.02}
[SH stage 1] cfg 03/18 | CV-MSE=0.355146 | {'hidden_layers': (256, 128), 'dropout': 0.16, 'lr': 0.0013032581411905396, 'weight_decay': 7.108396118035575e-07, 'use_layernorm': True, 'n_ensemble': 7, 'xi_base': 0.02}
[SH stage 1] cfg 04/18 | CV-MSE=0.140132 | {'hidden_layers': (64, 64), 'dropout': 0.06, 'lr': 0.0017060831586461437, 

C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



================ NEXT DATA POINT (SUBMIT THIS) — <= 6 DECIMALS ================

x_next = [0.198714 0.096701 0.135299 1.       0.405497]

================ MODEL PREDICTION (SURROGATE) ================

mu(x_next)    = -1.204589
sigma(x_next) = 0.204722
qEI_MC        = 0.0

================ WHY THIS POINT (PCA + CLUSTER) ================

chosen_mode   = MAIN(qEI+cluster+PCA)
source        = pca_sobol
effective_xi  = 0.005507
adaptive_min_dist = 0.004000
nearest_dist  = 0.367982
PCA: k= 4 var_threshold= 0.92
PCA explained var ratio: [0.362353, 0.266527, 0.171879, 0.127785, 0.071457]
cluster(best): id= 0 centroid= [0.32806, 0.174196, 0.576102, 0.856191, 0.028569]
cluster summary:
  - {'cluster': 0, 'n': 8, 'y_mean': -0.7123274344353702, 'y_max': -0.29318472382773614, 'best_x': [0.39532, 0.238886, 0.702458, 0.805605, 0.0]}
  - {'cluster': 1, 'n': 5, 'y_mean': -2.099246656530861, 'y_max': -1.24704893, 'best_x': [0.78495809, 0.91068235, 0.7081201, 0.95922543, 0.0049115]}
  - {'cluster': 2,